# AnyTimeLLaMa — LOCAL train

**WARNING**: 8B base model needs ≥40 GB VRAM. For 8GB local GPU, use a smaller variant (Llama-3.2-1B or 3B).

For 8B training use `train_colab.ipynb` on Colab Pro (A100/L4).

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))
%cd $REPO_ROOT

from shared import load_env
load_env()
print('HF_USER:', os.environ.get('HF_USER'))

## 2. Pick model + train

8B requires ~40GB VRAM. Use 1B / 3B for ≤8GB GPUs.

In [ ]:
# === EDIT MODEL HERE ===
BASE_MODEL = 'meta-llama/Llama-3.2-1B'   # or '3B' or 'Meta-Llama-3-8B'
EXIT_LAYERS = [4, 8, 12]                  # adjust per model depth
MAX_TRAIN_SAMPLES = 1000                  # smoke test
EPOCHS = 1

# Local training entry — uses finetune_ee.py with config
config_text = f'''model_name_or_path = {BASE_MODEL}
dataset_name = allenai/c4
dataset_config_name = en
text_column = text
output_dir = ./AnyTimeLLaMa/ckpts/local-{BASE_MODEL.split("/")[-1]}
max_train_samples = {MAX_TRAIN_SAMPLES}
max_val_samples = 100
max_seq_length = 1024
per_device_train_batch_size = 1
per_device_eval_batch_size = 1
gradient_accumulation_steps = 16
learning_rate = 1e-4
num_train_epochs = {EPOCHS}
logging_steps = 10
save_steps = 500
eval_steps = 500
seed = 42
torch_dtype = bfloat16
padding_side = right
report_to = ["none"]
overwrite_output_dir = true
exit_layer_indices = {EXIT_LAYERS}
exit_loss_weights = [0.4, 0.6, 0.8]
init_exit_from_base = true
exit_confidence_threshold = 0.9
'''

config_path = '/tmp/ee_config_local'
with open(config_path, 'w') as f:
    f.write(config_text)
print(config_text)

In [ ]:
%cd AnyTimeLLaMa
!python finetune_ee.py --config $config_path

## 3. Verify

In [ ]:
from pathlib import Path
out = Path(f'ckpts/local-{BASE_MODEL.split("/")[-1]}')
for f in sorted(out.glob('*')):
    print(' ', f.name)